In [ ]:


import numpy as np 
import pandas as pd 

# Input data files are available in the read-only 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
print(os.listdir('/kaggle/input'))

In [ ]:
import os

# walk through to find the real folder
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.csv' in files:
        base_path = root
        break

print("Found base path:", base_path)
print("Files:", os.listdir(base_path))

In [ ]:
import pandas as pd

train = pd.read_csv(f'{base_path}/train.csv')
print("Shape:", train.shape)
print("\nColumns:")
for c in train.columns:
    print(" -", c)

train.head()

In [ ]:
import numpy as np

# drop the meaningless index column
train = train.drop(columns=['Unnamed: 0'])

# check how much of the data is invalid
print("Total rows:", len(train))
print("\nBiomass summary stats:")
print(train['biomass'].describe())

print("\nRows with biomass == -9999:", (train['biomass'] == -9999).sum())
print("Rows with biomass < 0 (any negative):", (train['biomass'] < 0).sum())

# check missing values across all columns
print("\nMissing values per column:")
print(train.isnull().sum())

# check if qa_palsar is actually constant
print("\nUnique qa_palsar values:", train['qa_palsar_xxxx'].nunique())

In [ ]:
# look at the non-(-9999) negative values
odd_negatives = train[(train['biomass'] < 0) & (train['biomass'] != -9999)]
print("Count of odd negative biomass values:", len(odd_negatives))
print(odd_negatives['biomass'].describe())

# filter to valid rows: biomass > 0, and drop any remaining nulls
train_clean = train[train['biomass'] > 0].dropna()
print("\nClean training set shape:", train_clean.shape)
print("\nClean biomass stats:")
print(train_clean['biomass'].describe())

In [ ]:
train_clean = train_clean.copy()

# NDVI: (NIR - Red) / (NIR + Red), using Sentinel-2 bands
b04 = train_clean['b04_s2.cdse.quarterly_xxxx0401_xxxx0631']  # red
b08 = train_clean['b08_s2.cdse.quarterly_xxxx0401_xxxx0631']  # NIR
train_clean['ndvi'] = (b08 - b04) / (b08 + b04 + 1e-6)  # +eps avoids div-by-zero

# encode categoricals as integer codes (tree models handle this fine)
train_clean['bioregion'] = train_clean['bioregion'].astype('category')
train_clean['forest_type'] = train_clean['forest_type'].astype('category')

print(train_clean[['ndvi', 'bioregion', 'forest_type']].describe(include='all'))
print("\nNDVI range check - any inf/nan?", train_clean['ndvi'].isin([np.inf, -np.inf]).sum(), train_clean['ndvi'].isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PowerTransformer
from sklearn.metrics import mean_squared_error

# split by tile_id so validation tiles are never seen in training
unique_tiles = train_clean['tile_id'].unique()
train_tiles, val_tiles = train_test_split(unique_tiles, test_size=0.2, random_state=42)

tr = train_clean[train_clean['tile_id'].isin(train_tiles)]
val = train_clean[train_clean['tile_id'].isin(val_tiles)]
print("Train rows:", len(tr), "Val rows:", len(val))

feature_cols = [
    'b02_s2.cdse.quarterly_xxxx0401_xxxx0631',
    'b04_s2.cdse.quarterly_xxxx0401_xxxx0631',
    'b08_s2.cdse.quarterly_xxxx0401_xxxx0631',
    'obs_s2.cdse.quarterly_xxxx0401_xxxx0631',
    'HH_dB_palsar_xxxx', 'HV_dB_palsar_xxxx',
    'angle_palsar_xxxx', 'qa_palsar_xxxx',
    'vv.sentinel1_xxxx07', 'vh.sentinel1_xxxx07',
    'ndvi'
]

X_tr, y_tr = tr[feature_cols], tr['biomass']
X_val, y_val = val[feature_cols], val['biomass']

# --- Baseline: linear regression with Yeo-Johnson transform on target ---
pt = PowerTransformer(method='yeo-johnson')
y_tr_transformed = pt.fit_transform(y_tr.values.reshape(-1,1)).ravel()

lr = LinearRegression()
lr.fit(X_tr, y_tr_transformed)
pred_val_transformed = lr.predict(X_val)
pred_val = pt.inverse_transform(pred_val_transformed.reshape(-1,1)).ravel()

baseline_rmse = np.sqrt(mean_squared_error(y_val, pred_val))
print(f"Baseline (Linear + Yeo-Johnson) RMSE: {baseline_rmse:.3f}")

In [ ]:
import lightgbm as lgb

X_tr_lgb = X_tr.copy()
X_val_lgb = X_val.copy()

train_data = lgb.Dataset(X_tr_lgb, label=y_tr)
val_data = lgb.Dataset(X_val_lgb, label=y_val, reference=train_data)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'verbosity': -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

pred_val_lgb = model.predict(X_val_lgb, num_iteration=model.best_iteration)
lgb_rmse = np.sqrt(mean_squared_error(y_val, pred_val_lgb))
print(f"\nLightGBM RMSE: {lgb_rmse:.3f}  (vs your linear baseline: {baseline_rmse:.3f})")

In [ ]:
test = pd.read_csv(f'{base_path}/test.csv')
print("Test shape:", test.shape)
print("\nTest columns:")
for c in test.columns:
    print(" -", c)

test.head()

In [ ]:
sample_sub = pd.read_csv(f'{base_path}/sample_submission.csv')
print("Sample submission shape:", sample_sub.shape)
print(sample_sub.head())

# check if test.csv already has an 'Id' or 'row_id' style column
print("\nDoes test.csv have anything matching sample_submission Ids?")
print(test.columns.tolist())

In [ ]:
# Prepare full training data (100% of train_clean, not just 80%) 
X_full = train_clean[feature_cols]
y_full = train_clean['biomass']

full_train_data = lgb.Dataset(X_full, label=y_full)

# retrain using the same params, but without early stopping (no held-out set now)
# use best_iteration from before as a reasonable round count, with a small safety margin
final_model = lgb.train(
    params,
    full_train_data,
    num_boost_round=model.best_iteration  # reuse the round count that worked well
)

print("Final model trained on full dataset:", X_full.shape)

In [ ]:
# prepare test features exactly like training features
test_clean = test.dropna(subset=feature_cols[:-1]).copy()  # exclude 'ndvi' since we compute it
print("Test rows before NDVI/cleaning:", len(test), " -> after dropping NaNs:", len(test_clean))

b04_test = test_clean['b04_s2.cdse.quarterly_xxxx0401_xxxx0631']
b08_test = test_clean['b08_s2.cdse.quarterly_xxxx0401_xxxx0631']
test_clean['ndvi'] = (b08_test - b04_test) / (b08_test + b04_test + 1e-6)

test_clean['bioregion'] = test_clean['bioregion'].astype('category')
test_clean['forest_type'] = test_clean['forest_type'].astype('category')

X_test = test_clean[feature_cols]
test_preds = final_model.predict(X_test)

print("Predictions generated:", len(test_preds))
print("Prediction stats — min:", test_preds.min(), "max:", test_preds.max(), "mean:", test_preds.mean())

In [ ]:
# find which row_ids got dropped
missing_ids = set(test['row_id']) - set(test_clean['row_id'])
print("Missing row_ids:", missing_ids)

# look at what's actually wrong with those rows
missing_rows = test[test['row_id'].isin(missing_ids)]
print(missing_rows[feature_cols[:-1]])  

In [ ]:
# rebuild test_clean WITHOUT dropping any rows, impute missing values instead
test_full = test.copy()

# fill missing predictor values with the median from training data
for col in feature_cols[:-1]:  # exclude ndvi, computed after
    if test_full[col].isnull().any():
        median_val = train_clean[col].median()
        test_full[col] = test_full[col].fillna(median_val)
        print(f"Filled {test_full[col].isnull().sum()} remaining NaNs in {col} with median {median_val:.3f}")

# recompute ndvi safely
b04_all = test_full['b04_s2.cdse.quarterly_xxxx0401_xxxx0631']
b08_all = test_full['b08_s2.cdse.quarterly_xxxx0401_xxxx0631']
test_full['ndvi'] = (b08_all - b04_all) / (b08_all + b04_all + 1e-6)

test_full['bioregion'] = test_full['bioregion'].astype('category')
test_full['forest_type'] = test_full['forest_type'].astype('category')

X_test_full = test_full[feature_cols]
test_preds_full = final_model.predict(X_test_full)

print("\nTotal predictions:", len(test_preds_full), "— should equal 45019")
print("Prediction stats — min:", test_preds_full.min(), "max:", test_preds_full.max(), "mean:", test_preds_full.mean())

In [ ]:
# build submission dataframe
submission = pd.DataFrame({
    'Id': test_full['row_id'],
    'Expected': test_preds_full
})

print("Submission shape:", submission.shape)
print(submission.head())

# sanity check: does it match sample_submission's Ids 
ids_match = (submission['Id'].values == sample_sub['Id'].values).all()
print("\nIds match sample_submission exactly (same order):", ids_match)

# check for negative predictions 
print("Any negative predictions?", (submission['Expected'] < 0).sum())